# W6 · Day 1 — Building Naive RAG from Scratch

**~90 minutes · in-class demo · Jupyter notebook**

Today's session builds a complete naive RAG pipeline. Every stage is a
separate cell so you can see exactly what's happening.

This is **Day 1** — Topics 1, 2, 3 from the deck (Why RAG · Anatomy · Build).

**Corpus:** 10 short paragraphs about hot beverages. Small enough to read at
a glance. Diverse enough that retrieval must discriminate. This corpus stays
with us in Day 2 too.

**Cost per full run:** ~$0.002. Cheap.

**Notebook flow:**
- Cells 1-2: Setup + corpus
- Cells 3-3.5: Chunking (with a comparison of strategies)
- Cells 4-5.5: Embedding + similarity (with a playground)
- Cells 6-7: Retrieval + prompt construction
- Cells 8-8.5: Generation (and system prompt variations)
- Cell 9: Break it (4 failure modes to discuss)

---

## Cell 1 — Setup

Import what we need and confirm the OpenAI key is set. If this cell fails, fix
the environment before continuing — nothing else will work.

### Why are we doing this?

Before building the RAG pipeline, we need to prepare the Python environment and the OpenAI connection that the later cells will use.

This cell does not perform RAG yet. It only prepares the tools required for the RAG experiment.

### What does this cell prepare?

1. `os` — used to check whether the OpenAI API key is available.
2. `numpy` — used later for numerical calculations involving embedding vectors.
3. `OpenAI` — used to communicate with OpenAI's API.
4. `client` — creates the OpenAI client that will be used for embeddings and answer generation.
5. `EMBED_MODEL` — specifies the model used to convert text into numerical embeddings.
6. `CHAT_MODEL` — specifies the model used later to generate the final answer.

### Important distinction

Two different models are used for two different jobs:

- `text-embedding-3-small` → converts text into numbers (embeddings)
- `gpt-4o-mini` → generates the final answer

### API key check

The `assert` statement checks that `OPENAI_API_KEY` exists before we continue. If the key is missing, the notebook stops immediately because the later OpenAI API calls would not work.

### What comes out of this cell?

If everything is configured correctly, the output is:

`Setup ok. Ready to build RAG.`

This means the environment is ready for the next stage: creating the document corpus.

In [2]:
import os
import numpy as np
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"

print("Setup ok. Ready to build RAG.")

Setup ok. Ready to build RAG.


---

## Cell 2 — The corpus

10 short paragraphs about hot beverages: coffee, tea, hot chocolate, and
milk-based drinks. Diverse enough that retrieval has to discriminate — a
query about lattes shouldn't match hot chocolate chunks.

### What is a corpus?

A corpus is the collection of documents or text that our RAG system is allowed to use as its knowledge source.

For this Week 6 concept demonstration, the corpus contains 10 short paragraphs about hot beverages such as coffee, tea, hot chocolate, and latte.

### Why are we using this corpus?

This is a deliberately small and simple corpus. The purpose of Week 6 Track A is to understand how RAG works, not to work with a real enterprise document collection.

The documents are also related to each other but contain different information. This allows us to test whether retrieval can distinguish between similar topics.

### Structure of each document

Each document is stored as a Python dictionary with two fields:

- `id` — a unique identifier for the document
- `text` — the actual information contained in the document

For example:

`coffee_espresso` identifies the document, while its `text` contains information about espresso.

### Why do we need document IDs?

The IDs allow us to identify where retrieved information came from. Later, when RAG retrieves chunks, the source ID can be included with the answer.

### What happens in this cell?

The 10 documents are stored in the `documents` list. The code then prints the number of documents and the character length of each document.

### What comes out of this cell?

We should see:

`Loaded 10 documents.`

followed by the ID and character count of each of the 10 documents.

### Position in the RAG pipeline

At this stage we have only the source documents:

Documents → **Chunking** → Embeddings → Similarity → Retrieval → Prompt → Answer

The next cell will split these documents into smaller chunks.

In [3]:
documents = [
    # Coffee
    {"id": "coffee_espresso", "text": (
        "Espresso is a concentrated form of coffee made by forcing hot water "
        "under about 9 bars of pressure through finely ground coffee beans. "
        "A single shot is typically 25 to 30 millilitres and takes 25 to 30 "
        "seconds to extract. Espresso forms the base of drinks like the "
        "latte, cappuccino, and americano."
    )},
    {"id": "coffee_beans", "text": (
        "Coffee beans come primarily from two species: Arabica and Robusta. "
        "Arabica accounts for about 60 percent of world production and is "
        "prized for its smoother, more nuanced flavour. Robusta contains "
        "roughly twice as much caffeine and has a stronger, more bitter taste. "
        "Most commercial espresso blends mix the two."
    )},
    {"id": "coffee_brewing", "text": (
        "Pour-over coffee uses a filter cone to drip near-boiling water "
        "through medium-ground coffee. It typically brews for three to four "
        "minutes and produces a clean, light-bodied cup. French press coffee, "
        "in contrast, steeps coarse grounds directly in hot water for four "
        "minutes before pressing, producing a heavier, oil-rich cup."
    )},
    # Tea
    {"id": "tea_green", "text": (
        "Green tea is made from unoxidised leaves of Camellia sinensis. It is "
        "steeped in water at around 70 to 80 degrees Celsius for one to three "
        "minutes. Hotter water or longer steeping produces a bitter, astringent "
        "cup. Green tea is high in an antioxidant called EGCG."
    )},
    {"id": "tea_black", "text": (
        "Black tea comes from fully oxidised Camellia sinensis leaves. It is "
        "brewed with water at or near boiling — 95 to 100 degrees Celsius — "
        "for three to five minutes. Popular varieties include Assam, Darjeeling, "
        "and Ceylon. Black tea typically contains more caffeine than green tea."
    )},
    {"id": "tea_oolong", "text": (
        "Oolong tea is partially oxidised, sitting between green and black tea "
        "in strength and colour. It is brewed at 85 to 95 degrees Celsius for "
        "two to four minutes. Oolong leaves are often rolled and can be re-steeped "
        "several times, with each infusion revealing different flavour notes."
    )},
    # Hot chocolate
    {"id": "chocolate_traditional", "text": (
        "Traditional hot chocolate is made from melted dark chocolate stirred "
        "into hot milk. The ratio is usually 30 to 50 grams of chocolate per "
        "200 millilitres of milk. Whisking prevents the chocolate from settling. "
        "Some recipes add a pinch of chilli or cinnamon for warmth."
    )},
    {"id": "chocolate_powder", "text": (
        "Instant hot chocolate uses cocoa powder mixed with sugar, milk powder, "
        "and stabilisers. Adding hot water dissolves the mix in seconds. It is "
        "cheaper and faster than the traditional method but has a thinner mouthfeel "
        "and less intense chocolate flavour."
    )},
    {"id": "chocolate_history", "text": (
        "Hot chocolate originated with the Maya and Aztec civilisations, who "
        "drank it cold and bitter, spiced with chilli. Europeans encountered "
        "cacao in the 16th century and gradually sweetened the drink and "
        "served it hot. It remained a luxury until industrial cocoa processing "
        "made it affordable in the 19th century."
    )},
    # Milk-based drinks
    {"id": "milk_latte", "text": (
        "A caffè latte is made with one shot of espresso and around 200 "
        "millilitres of steamed milk topped with a thin layer of microfoam. "
        "The ratio is roughly one part espresso to five parts milk. A "
        "cappuccino uses the same espresso base but has equal parts milk and "
        "foam, giving it a lighter, airier texture."
    )},
]

print(f"Loaded {len(documents)} documents.")
for d in documents:
    print(f"  {d['id']:25s}  {len(d['text']):3d} chars")

Loaded 10 documents.
  coffee_espresso            298 chars
  coffee_beans               310 chars
  coffee_brewing             324 chars
  tea_green                  262 chars
  tea_black                  277 chars
  tea_oolong                 281 chars
  chocolate_traditional      267 chars
  chocolate_powder           251 chars
  chocolate_history          309 chars
  milk_latte                 301 chars


In [4]:
for i in documents:
    print(i)

{'id': 'coffee_espresso', 'text': 'Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.'}
{'id': 'coffee_beans', 'text': 'Coffee beans come primarily from two species: Arabica and Robusta. Arabica accounts for about 60 percent of world production and is prized for its smoother, more nuanced flavour. Robusta contains roughly twice as much caffeine and has a stronger, more bitter taste. Most commercial espresso blends mix the two.'}
{'id': 'coffee_brewing', 'text': 'Pour-over coffee uses a filter cone to drip near-boiling water through medium-ground coffee. It typically brews for three to four minutes and produces a clean, light-bodied cup. French press coffee, in contrast, steeps coarse grounds directly in hot water for four minutes before press

---

## Cell 3 — Chunking (sliding window)

Split each document into overlapping windows. We use the simplest strategy:
**sliding window of 200 characters with 40 characters of overlap.**

### Why are we doing this?

Our corpus currently contains 10 complete documents. Before we create embeddings and perform retrieval, we need to divide these documents into smaller pieces called **chunks**.

A user question usually needs only a small part of a document. If we keep an entire document as one large unit, retrieval becomes less precise. By breaking documents into smaller chunks, the RAG system can later retrieve the specific pieces of information that are most relevant to a question.

### What is chunking?

**Chunking means dividing a larger piece of text into smaller pieces.**

For example:

```text
Whole document
────────────────────────────────────
              ↓
       Chunk 1
       Chunk 2
       Chunk 3
```

In this notebook, we use a **sliding-window** approach.

### What is a sliding window?

The function looks at a fixed number of characters at a time and then moves forward through the document.

The function uses:

- `size = 200` → each chunk contains up to 200 characters.
- `overlap = 40` → the next chunk repeats the last 40 characters of the previous chunk.

Conceptually:

```text
Chunk 1
[----------------------------------------]
                    200 characters

                         ↓ move forward
                    ← 40 characters →
                         
                    Chunk 2
                    [----------------------------------------]
```

The overlapping area creates a small connection between neighbouring chunks. This helps reduce the chance of separating related information exactly at a chunk boundary.

### What does `chunk_text()` do?

The `chunk_text()` function:

1. Receives the text of one document.
2. Checks whether the text is already smaller than the requested chunk size.
3. If it is small enough, returns the complete text as one chunk.
4. Otherwise, creates chunks using a sliding window.
5. Moves the window forward while keeping the specified overlap.
6. Returns a list containing all the chunks.

### What happens after the function is defined?

The code applies `chunk_text()` to every document in our 10-document corpus.

Each generated chunk is stored with three pieces of information:

- `chunk_id` → uniquely identifies the chunk.
- `source_id` → identifies the original document from which the chunk came.
- `text` → contains the actual chunk text.

For example:

```text
chunk_id  → coffee_espresso#0
source_id → coffee_espresso
text      → actual text of the chunk
```

This source information will be useful later when we retrieve chunks and want to know which document they came from.

### What comes out of this cell?

The cell produces one flat list called `all_chunks`.

Instead of having only:

```text
10 complete documents
```

we now have:

```text
10 documents
      ↓
chunking
      ↓
many smaller chunks
```

The cell prints the total number of chunks and shows the first few chunks so that we can inspect the result.

### Important point

This cell **does not create embeddings yet**.

At the end of this cell we have:

```text
text chunks
```

The next embedding stage will convert these chunks into numerical vectors.

### Position in the RAG pipeline

```text
Documents
    ↓
[CURRENT STEP — CHUNKING]
    ↓
Chunks
    ↓
Embeddings
    ↓
Similarity
    ↓
Top-K Retrieval
    ↓
Prompt
    ↓
LLM Answer
```

In [5]:
def chunk_text(text, size=200, overlap=40):
    """Sliding window over characters."""
    if len(text) <= size:
        return [text]
    chunks, i = [], 0
    while i < len(text):
        end = min(i + size, len(text))
        chunks.append(text[i:end])
        if end == len(text):
            break
        i = end - overlap
    return chunks

# Chunk every document; build a flat list with source pointer
all_chunks = []
for doc in documents:
    for chunk_idx, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "chunk_id":  f"{doc['id']}#{chunk_idx}",
            "source_id": doc["id"],
            "text":      chunk,
        })

print(f"Total chunks from {len(documents)} documents: {len(all_chunks)}\n")
for c in all_chunks[:8]:  # show first 8 to avoid wall of text
    marker = "…" if len(c["text"]) == 200 else " "
    print(f"  {c['chunk_id']:30s} [{len(c['text']):3d} chars] {c['text'][:60]}{marker}")
print(f"  ... and {len(all_chunks) - 8} more chunks")

Total chunks from 10 documents: 20

  coffee_espresso#0              [200 chars] Espresso is a concentrated form of coffee made by forcing ho…
  coffee_espresso#1              [138 chars] y 25 to 30 millilitres and takes 25 to 30 seconds to extract 
  coffee_beans#0                 [200 chars] Coffee beans come primarily from two species: Arabica and Ro…
  coffee_beans#1                 [150 chars] e nuanced flavour. Robusta contains roughly twice as much ca 
  coffee_brewing#0               [200 chars] Pour-over coffee uses a filter cone to drip near-boiling wat…
  coffee_brewing#1               [164 chars] light-bodied cup. French press coffee, in contrast, steeps c 
  tea_green#0                    [200 chars] Green tea is made from unoxidised leaves of Camellia sinensi…
  tea_green#1                    [102 chars] or longer steeping produces a bitter, astringent cup. Green  
  ... and 12 more chunks


In [6]:
print(all_chunks)

[{'chunk_id': 'coffee_espresso#0', 'source_id': 'coffee_espresso', 'text': 'Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3'}, {'chunk_id': 'coffee_espresso#1', 'source_id': 'coffee_espresso', 'text': 'y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.'}, {'chunk_id': 'coffee_beans#0', 'source_id': 'coffee_beans', 'text': 'Coffee beans come primarily from two species: Arabica and Robusta. Arabica accounts for about 60 percent of world production and is prized for its smoother, more nuanced flavour. Robusta contains roug'}, {'chunk_id': 'coffee_beans#1', 'source_id': 'coffee_beans', 'text': 'e nuanced flavour. Robusta contains roughly twice as much caffeine and has a stronger, more bitter taste. Most commercial espresso blends mix the two.'}, {'chunk

---

## Cell 3.5 — Chunking strategies compared

Sliding-window chunking is the simplest, but it's not the only choice. Let's
compare three strategies on the same document — `coffee_espresso` — and see
the difference in cleanliness.

**Watch:** notice how sentence-based chunks are semantically cleaner, but you
get variable-size chunks. Sliding window gives uniform sizes but cuts through
meaning. Paragraph-based is even cleaner but our corpus is one paragraph per
doc — so it produces one "chunk" per doc, potentially too large for a good
retrieval prompt.


### Why are we doing this?

In Cell 3, we used a sliding-window method to divide documents into chunks.

Here, we compare three different ways of creating chunks from the same document:

1. Sliding window
2. Sentence-based chunking
3. Paragraph-based chunking

The purpose is to understand the trade-offs between different chunking strategies.

### 1. Sliding window

The sliding-window method uses a fixed chunk size and overlap.

In this notebook:

- Chunk size = 200 characters
- Overlap = 40 characters

For our `coffee_espresso` document, this produced 2 chunks.

The advantage is that chunk sizes are predictable.

The disadvantage is that the method does not understand language structure. It can cut words or sentences in the middle, as we can see in the output.

### 2. Sentence-based chunking

The sentence-based method creates one chunk for each sentence.

For our sample document, it produced 3 chunks, with each sentence remaining intact.

This gives cleaner semantic units, but the chunk sizes are different:

- 134 characters
- 86 characters
- 76 characters

Therefore, some sentences could be too small or, in other documents, too large.

### 3. Paragraph-based chunking

The paragraph-based method creates one chunk for each paragraph.

Our sample document contains one paragraph, so the entire 298-character document became one chunk.

This preserves the context within the paragraph, but paragraphs in real documents can sometimes be too large to use as individual chunks.

### Comparison

| Strategy | Advantage | Limitation |
|---|---|---|
| Sliding window | Predictable chunk sizes | Can cut words or sentences |
| Sentence | Keeps sentences intact | Variable chunk sizes |
| Paragraph | Preserves semantic context | A paragraph may be too large |

### Key takeaway

There is no single perfect chunking strategy.

Real systems can combine different strategies depending on the type of documents and the retrieval task.

For this Week 6 naive RAG exercise, we continue with the **sliding-window approach** so that we can focus on understanding the basic RAG pipeline.

### Position in the RAG pipeline

```text
Documents
    ↓
Chunking  ← We are comparing strategies here
    ↓
Embeddings
    ↓
Similarity
    ↓
Top-K Retrieval
    ↓
Prompt
    ↓
LLM Answer

In [7]:
import re

def chunk_by_sentence(text):
    """Split on sentence boundaries. Simpler than a proper NLP splitter."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]

def chunk_by_paragraph(text):
    """One chunk per paragraph."""
    return [p.strip() for p in text.split("\n\n") if p.strip()]

sample = documents[0]["text"]  # coffee_espresso
print(f"Sample document: {documents[0]['id']} ({len(sample)} chars)\n")
print(f"Text: {sample}\n")
print("═" * 80)

sw_chunks = chunk_text(sample)
print(f"\n1. SLIDING WINDOW (size=200, overlap=40) → {len(sw_chunks)} chunks")
for i, c in enumerate(sw_chunks):
    print(f"    [{i}] ({len(c):3d} chars) {c!r}")

sent_chunks = chunk_by_sentence(sample)
print(f"\n2. BY SENTENCE → {len(sent_chunks)} chunks")
for i, c in enumerate(sent_chunks):
    print(f"    [{i}] ({len(c):3d} chars) {c!r}")

para_chunks = chunk_by_paragraph(sample)
print(f"\n3. BY PARAGRAPH → {len(para_chunks)} chunks")
for i, c in enumerate(para_chunks):
    print(f"    [{i}] ({len(c):3d} chars) {c!r}")

print("\n" + "═" * 80)
print("Discussion:")
print("  - Sliding window: uniform sizes but cuts words/sentences")
print("  - Sentence: clean units but variable size (some too small)")
print("  - Paragraph: cleanest semantically but often too large per chunk")
print("  - Real systems combine strategies. We use sliding window for now.")

Sample document: coffee_espresso (298 chars)

Text: Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.

════════════════════════════════════════════════════════════════════════════════

1. SLIDING WINDOW (size=200, overlap=40) → 2 chunks
    [0] (200 chars) 'Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3'
    [1] (138 chars) 'y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.'

2. BY SENTENCE → 3 chunks
    [0] (134 chars) 'Espresso is a concentrated form of coffee made by forcing hot water under about 9 

---

## Cell 4 — Embeddings

Convert each chunk to a vector using OpenAI's `text-embedding-3-small` model.
Every chunk becomes a **1536-dimensional vector**.

**Look at the shape of the output.** An embedding is just a list of floats.
You cannot read them by eye. That's the point — the model has compressed
meaning into coordinates.

### Why are we doing this?

The chunks created in the previous step are still ordinary text. A computer cannot easily compare the meaning of two pieces of text just by looking at their characters.

An **embedding** converts text into a numerical vector so that we can later calculate how similar different pieces of text are.

### What happens in this cell?

We use the OpenAI `text-embedding-3-small` model to convert every chunk into an embedding.

Our corpus contains:

```text
10 documents
     ↓
20 chunks
     ↓
20 embeddings

Each embedding contains 1536 floating-point values.
Conceptually:
Text chunk
    ↓
Embedding model
    ↓
[0.01, -0.03, 0.08, ...]
    ↑
1536 numbers

Why do we use embed_batch()?
Instead of sending each chunk to the API separately, we send the list of chunks together in one embedding API call.
The function returns one embedding vector for each input chunk.
What does zip(all_chunks, vectors) do?
It pairs each chunk with its corresponding embedding.
Each chunk therefore becomes conceptually:
{
    chunk_id,
    source_id,
    text,
    vector
}

The vector contains the numerical embedding for that particular chunk.
Can we understand the individual numbers?
No.
An individual value such as:
-0.03256

does not have a human-readable meaning such as "coffee" or "espresso".
The useful information comes from the overall pattern of the 1536 values and from comparing one vector with another.
What did this cell produce?
The cell successfully created:
- 20 embeddings
- 1536 dimensions per embedding
The next step is to compare these vectors using similarity.
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings  ← CURRENT STEP
    ↓
Similarity  ← NEXT STEP
    ↓
Top-K Retrieval
    ↓
Prompt
    ↓
LLM Answer

Key takeaway
Embeddings turn text into numerical representations that allow a RAG system to compare the semantic similarity between a user's question and stored text chunks.

In [8]:
def embed_batch(texts):
    """One API call, list of 1536-dim vectors back."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]

chunk_texts = [c["text"] for c in all_chunks]
vectors = embed_batch(chunk_texts)

for chunk, vec in zip(all_chunks, vectors):
    chunk["vector"] = vec

print(f"Embedded {len(vectors)} chunks.")
print(f"Each embedding shape: {len(vectors[0])} dimensions")
print(f"\nFirst 8 dims of chunk 0 ({all_chunks[0]['chunk_id']}):")
print(f"  {vectors[0][:8]}")
print(f"\nFirst 8 dims of chunk 1 ({all_chunks[1]['chunk_id']}):")
print(f"  {vectors[1][:8]}")
print("\nTakeaway: embeddings are floats. You cannot read them by eye.")
print("But you CAN compute similarity between them — next cell.")

Embedded 20 chunks.
Each embedding shape: 1536 dimensions

First 8 dims of chunk 0 (coffee_espresso#0):
  [-0.032562255859375, -0.03851318359375, -0.003143310546875, -0.06591796875, 0.00930023193359375, -0.038299560546875, -0.029296875, 0.053619384765625]

First 8 dims of chunk 1 (coffee_espresso#1):
  [-0.038665771484375, -0.00775146484375, 0.025482177734375, -0.07611083984375, -0.004619598388671875, -0.0208740234375, 0.01171112060546875, 0.045928955078125]

Takeaway: embeddings are floats. You cannot read them by eye.
But you CAN compute similarity between them — next cell.


In [9]:
def embed_batch_1(texts):
    """One API call, list of 1536-dim vectors back."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return resp
    # return [item.embedding for item in resp.data]
    
print(sample)
l1= sample.split(" ")
print(l1, "\n", len(l1))

print(embed_batch_1(sample))
#l2= embed_batch(sample)[0]
#print(l2, "\n", len(l2))

Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.
['Espresso', 'is', 'a', 'concentrated', 'form', 'of', 'coffee', 'made', 'by', 'forcing', 'hot', 'water', 'under', 'about', '9', 'bars', 'of', 'pressure', 'through', 'finely', 'ground', 'coffee', 'beans.', 'A', 'single', 'shot', 'is', 'typically', '25', 'to', '30', 'millilitres', 'and', 'takes', '25', 'to', '30', 'seconds', 'to', 'extract.', 'Espresso', 'forms', 'the', 'base', 'of', 'drinks', 'like', 'the', 'latte,', 'cappuccino,', 'and', 'americano.'] 
 52


CreateEmbeddingResponse(data=[Embedding(embedding=[-0.041168212890625, -0.041839599609375, -0.006725311279296875, -0.05902099609375, 0.01885986328125, -0.031097412109375, -0.0355224609375, 0.0531005859375, 0.0543212890625, -0.058135986328125, 0.03741455078125, 0.02099609375, -0.041107177734375, -0.0049591064453125, -0.0321044921875, 0.0003409385681152344, 0.0010633468627929688, -0.03179931640625, 0.0062103271484375, 0.00893402099609375, -0.047607421875, -0.01702880859375, -0.01151275634765625, 0.045745849609375, 0.02423095703125, 0.006175994873046875, -0.015899658203125, -0.01107025146484375, -0.0130157470703125, -0.00962066650390625, -0.009857177734375, -0.00977325439453125, 0.0015592575073242188, -0.0124969482421875, 0.0382080078125, 0.00274658203125, -0.01898193359375, 0.0254669189453125, -0.02423095703125, 0.0052032470703125, -0.000946044921875, -0.0199737548828125, -0.036834716796875, -0.06842041015625, 0.024322509765625, -0.040802001953125, -0.0015764236450195312, -0.041259765625

---

## Cell 5 — Cosine similarity

Given two vectors, cosine similarity is a number that tells you how similar
they are in direction. Higher = more similar.

**Rule of thumb** with `text-embedding-3-small`:
- 0.7+ → very similar meaning
- 0.5 to 0.7 → related
- 0.3 to 0.5 → weakly related
- Below 0.3 → probably unrelated

### Why are we doing this?

We now have an embedding for every document chunk.

When a user asks a question, we also convert the question into an embedding.

We now need a way to compare the question embedding with each stored chunk embedding and determine which chunks are most relevant.

This cell uses **cosine similarity** for that comparison.

### Step 1 — Create a query embedding

Our query is:

```text
how is espresso made?
The query is converted into a 1536-dimensional embedding using the same embedding model used for the document chunks.
Question
    ↓
Embedding model
    ↓
Query vector
Step 2 — Calculate cosine similarity
The cosine() function compares two vectors.
In this experiment:
Query vector
     ↕
Chunk vector
A higher cosine similarity score means that the two vectors are more similar in the embedding space.
The score is used for ranking. It should not be interpreted as a percentage of correctness or as a probability.
Step 3 — Compare the query with every chunk
The code calculates a similarity score between the query vector and every one of our 20 chunk vectors.
Conceptually:
Question
    ↓
Query embedding
    ↓
Compare with all 20 chunk embeddings
    ↓
20 similarity scores
Step 4 — Sort the results
The chunks are sorted from the highest similarity score to the lowest.
For our query, the highest-ranked result was:
0.669  coffee_espresso#0
This makes sense because the chunk directly discusses how espresso is made.
The second result was:
0.575  coffee_espresso#1
Other related chunks, such as the latte and coffee-bean chunks, also received relatively high scores.
Important observation
Similarity does not mean that a chunk contains the exact answer.
For example, latte-related chunks were ranked highly because latte is related to espresso.
Therefore, naive semantic retrieval can return information that is related but not necessarily the most useful for answering the question.
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Question embedding
    ↓
Cosine similarity  ← CURRENT STEP
    ↓
Rank chunks
    ↓
Top-K retrieval   ← NEXT STEP
    ↓
Prompt
    ↓
LLM Answer
Key takeaway
Cosine similarity allows us to rank stored chunks according to how semantically similar they are to the user's question.
The next step will select only the highest-ranked chunks using Top-K retrieval.


In [10]:
def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

QUERY = "how is espresso made?"
query_vector = embed_batch([QUERY])[0]

scored = [(cosine(query_vector, c["vector"]), c) for c in all_chunks]
scored.sort(key=lambda pair: pair[0], reverse=True)

print(f"Query: {QUERY!r}\n")
print(f"  {'cosine':>7s}  {'chunk_id':<30s}  preview")
print(f"  {'------':>7s}  {'--------':<30s}  {'-------':<50s}")
for score, chunk in scored[:10]:
    print(f"  {score:>7.3f}  {chunk['chunk_id']:<30s}  {chunk['text'][:50]}...")

Query: 'how is espresso made?'

   cosine  chunk_id                        preview
   ------  --------                        -------                                           
    0.669  coffee_espresso#0               Espresso is a concentrated form of coffee made by ...
    0.575  coffee_espresso#1               y 25 to 30 millilitres and takes 25 to 30 seconds ...
    0.570  milk_latte#1                    espresso to five parts milk. A cappuccino uses the...
    0.565  milk_latte#0                    A caffè latte is made with one shot of espresso an...
    0.540  coffee_beans#1                  e nuanced flavour. Robusta contains roughly twice ...
    0.464  coffee_brewing#1                light-bodied cup. French press coffee, in contrast...
    0.429  coffee_brewing#0                Pour-over coffee uses a filter cone to drip near-b...
    0.377  coffee_beans#0                  Coffee beans come primarily from two species: Arab...
    0.375  chocolate_history#1             y an

---

## Cell 5.5 — Similarity playground

A tiny function that lets you type ANY query and see the top-K ranked. Try
different queries and build intuition for how cosine behaves.

The pre-defined queries below cover common cases: on-topic, cross-topic,
ambiguous, and out-of-scope. Look at the top scores for each — which cases
produce clean rankings? Which don't?

### Why are we doing this?

In the previous cell, we tested retrieval with one question:

```text
how is espresso made?

Here we test the same retrieval process with five different types of queries.
The purpose is to observe how naive semantic retrieval behaves when questions are specific, ambiguous, multi-faceted, cross-topic, or outside the corpus.
Query 1 — Cross-topic
how much caffeine is in tea?

The top three results were all tea-related chunks.
This shows that semantic retrieval can identify the general topic of a query.
However, semantic relevance does not guarantee that a retrieved chunk contains the exact fact requested by the user.
Query 2 — Very specific
what is a latte?

The highest-ranked result was milk_latte#0, which directly discusses what a caffè latte is.
This is an example of a relatively successful retrieval.
The espresso chunks also appear because espresso is related to latte.
Query 3 — Ambiguous
traditional recipe

The query does not specify which drink the user means.
The retriever returned chocolate-related chunks.
This demonstrates that a vague query can lead to weak or arbitrary retrieval because the system has no clarification step.
Query 4 — Multi-facet
origin of hot chocolate in the 16th century

The highest-ranked results were both from chocolate_history.
This shows that semantic retrieval can identify the appropriate part of the corpus when several related concepts are present in the query.
Query 5 — Out-of-scope
how do you brew beer?

Our corpus contains coffee, tea, and hot chocolate information, but no beer information.
Nevertheless, the retriever returned coffee and tea chunks.
This is an important limitation of naive retrieval:
The retriever returns the closest available chunks even when the corpus does not contain the answer.
It does not automatically know that the correct response should be "I don't have enough information."
Key takeaway
Similarity retrieval is useful for finding relevant-looking information, but a high similarity score does not guarantee that the retrieved chunk contains the correct answer.
This is especially important for:
- ambiguous questions,
- out-of-scope questions,
- questions requiring several facts,
- and questions where semantically related information is not sufficient.
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Question embedding
    ↓
Cosine similarity
    ↓
Rank chunks
    ↓
Similarity Playground  ← CURRENT EXPERIMENT
    ↓
Top-K retrieval
    ↓
Prompt
    ↓
LLM Answer

Key takeaway
Naive semantic retrieval can find related information, but it does not automatically know whether the retrieved information is sufficient or correct for the user's question.

In [11]:
def show_top_k(query, k=3):
    """Given a query, print the top-K chunks with their cosine scores."""
    q_vec = embed_batch([query])[0]
    scored = [(cosine(q_vec, c["vector"]), c) for c in all_chunks]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    print(f"\nQ: {query!r}")
    for i, (score, chunk) in enumerate(scored[:k], 1):
        print(f"  [{i}] {score:.3f}  {chunk['chunk_id']:<25s} {chunk['text'][:60]}...")

# Explore different query types:
show_top_k("how much caffeine is in tea?")           # cross-topic (tea + caffeine)
show_top_k("what is a latte?")                        # very specific
show_top_k("traditional recipe")                       # ambiguous (which drink?)
show_top_k("origin of hot chocolate in the 16th century")  # multi-facet
show_top_k("how do you brew beer?")                    # out-of-scope


Q: 'how much caffeine is in tea?'
  [1] 0.608  tea_black#1               . Popular varieties include Assam, Darjeeling, and Ceylon. B...
  [2] 0.472  tea_green#1               or longer steeping produces a bitter, astringent cup. Green ...
  [3] 0.453  tea_black#0               Black tea comes from fully oxidised Camellia sinensis leaves...

Q: 'what is a latte?'
  [1] 0.637  milk_latte#0              A caffè latte is made with one shot of espresso and around 2...
  [2] 0.502  coffee_espresso#1         y 25 to 30 millilitres and takes 25 to 30 seconds to extract...
  [3] 0.467  milk_latte#1              espresso to five parts milk. A cappuccino uses the same espr...

Q: 'traditional recipe'
  [1] 0.383  chocolate_powder#1        than the traditional method but has a thinner mouthfeel and ...
  [2] 0.311  chocolate_traditional#0   Traditional hot chocolate is made from melted dark chocolate...
  [3] 0.297  chocolate_traditional#1   . Whisking prevents the chocolate from settling. Some 

**Try your own.** Change the queries above or add new ones. Which queries
produce a clear winner (top-1 score much higher than top-2)? Which produce
many close scores (retrieval is uncertain)?

**Key observation:** even the out-of-scope query returns SOMETHING. Cosine
always returns the top-K by rank, even when nothing is truly relevant. This
is why we need thresholds and grounded prompts (next cells).

---

## Cell 6 — Top-K retrieval

Wrap the ranking into a clean function. K is a hyperparameter — we'll use
**K=3** for now. (Day 2 explores what happens when K changes.)

### Why are we doing this?

In the previous cells, we calculated cosine similarity between a question and every document chunk.

Here, we turn that process into a reusable `retrieve()` function.

The function finds the chunks that are most similar to the user's question and returns only the top `K` results.

### What does `retrieve()` do?

The function takes three inputs:

- `query` — the user's question
- `chunks` — the collection of chunks to search
- `k` — the number of results to return

The default value is:

```text
k = 3
Retrieval process
Question
    ↓
Create question embedding
    ↓
Compare with every chunk using cosine similarity
    ↓
Sort chunks by similarity
    ↓
Return the top K chunks
What happened in this experiment?
For the query:
how is espresso made?
the top three retrieved chunks were:
1. coffee_espresso#0  — cosine 0.669
2. coffee_espresso#1  — cosine 0.575
3. milk_latte#1       — cosine 0.570
The first two results are from the espresso document and are directly relevant to the question.
The third result is related to espresso but is less directly useful. This demonstrates that semantic retrieval can return related information rather than only information that is essential to the answer.
Why do we use Top-K?
Our corpus contains 20 chunks, but we do not want to send all 20 chunks to the LLM for every question.
Instead, we select the most relevant few chunks.
For example:
k = 3
means that only the three highest-ranked chunks are selected.
What does the returned result contain?
Each retrieved chunk keeps its original information and also receives its cosine similarity score.
Conceptually:
{
    chunk_id,
    source_id,
    text,
    vector,
    score
}
Important distinction
This function performs retrieval, not answer generation.
It finds information that may be useful for answering the question.
The LLM has not generated an answer yet.
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Question embedding
    ↓
Cosine similarity
    ↓
Rank chunks
    ↓
Top-K Retrieval  ← CURRENT STEP
    ↓
Prompt Construction  ← NEXT STEP
    ↓
LLM Answer
Key takeaway
Top-K retrieval selects a small set of potentially relevant chunks that can be provided to the LLM as context for generating the final answer.


In [12]:
def retrieve(query, chunks, k=3):
    q_vec = embed_batch([query])[0]
    scored = [(cosine(q_vec, c["vector"]), c) for c in chunks]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [{**c, "score": s} for s, c in scored[:k]]

top3 = retrieve(QUERY, all_chunks, k=3)

print(f"Top 3 chunks for query {QUERY!r}:\n")
for i, hit in enumerate(top3, 1):
    print(f"  [{i}] {hit['chunk_id']}  (cosine {hit['score']:.3f})")
    print(f"      {hit['text']}\n")

Top 3 chunks for query 'how is espresso made?':

  [1] coffee_espresso#0  (cosine 0.669)
      Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3

  [2] coffee_espresso#1  (cosine 0.575)
      y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.

  [3] milk_latte#1  (cosine 0.570)
      espresso to five parts milk. A cappuccino uses the same espresso base but has equal parts milk and foam, giving it a lighter, airier texture.



---

## Cell 7 — Prompt construction

Now the crucial step people underestimate: **what does the LLM actually see?**

We build a prompt with three parts:
1. System instruction (the LLM's job)
2. Retrieved chunks as context
3. User's question

**Print the exact prompt below.** This is everything the LLM has for this
query. No memory of your corpus. No lookup ability.

### Why are we doing this?

The retrieval stage has selected the top-K chunks that appear most relevant to the user's question.

However, the LLM cannot directly see Python variables such as `top3` or `all_chunks`.

We therefore need to convert the retrieved chunks into text and place them inside a prompt.

This is where the retrieval part of RAG is connected to the generation part.

### System instruction

The `SYSTEM` message gives the LLM rules for answering.

It tells the model to:

1. Use only the provided context.
2. Say plainly when the context does not contain the answer.
3. Cite the source ID after facts that it uses.

This helps keep the generated answer grounded in the retrieved context.

### Building the context

The `build_prompt()` function takes the retrieved chunks and converts them into readable text.

Each chunk is formatted as:

```text
[source_id]
chunk text
For example:
[coffee_espresso#0]
Espresso is a concentrated form of coffee...
The source ID is included so that the generated answer can identify where its information came from.
Building the user message
The user message contains two parts:
Context:
    retrieved chunks

---

Question:
    user's question
The separator helps distinguish the retrieved information from the actual question.
Important concept
The LLM does not automatically see our Python data.
The retrieved chunks must be explicitly inserted into the prompt.
The flow is:
Question
    ↓
Retrieve relevant chunks
    ↓
Build prompt containing those chunks
    ↓
Send prompt to LLM
    ↓
Generate answer
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Question embedding
    ↓
Cosine similarity
    ↓
Top-K retrieval
    ↓
Prompt Construction  ← CURRENT STEP
    ↓
LLM Answer  ← NEXT STEP
Key takeaway
Retrieval finds the information; prompt construction puts that information in a form the LLM can use to generate a grounded answer.


In [13]:
SYSTEM = (
    "You are a helpful assistant. Answer the user's question using ONLY the "
    "provided context. If the context does not contain the answer, say so "
    "plainly. Cite the source id in square brackets after any fact you use."
)

def build_prompt(question, retrieved, system=SYSTEM):
    context = "\n\n".join(
        f"[{hit['chunk_id']}]\n{hit['text']}"
        for hit in retrieved
    )
    user_msg = f"Context:\n{context}\n\n---\n\nQuestion: {question}"
    return system, user_msg

system_msg, user_msg = build_prompt(QUERY, top3)

print("═══ SYSTEM ═══")
print(system_msg)
print("\n═══ USER ═══")
print(user_msg)

═══ SYSTEM ═══
You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context does not contain the answer, say so plainly. Cite the source id in square brackets after any fact you use.

═══ USER ═══
Context:
[coffee_espresso#0]
Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3

[coffee_espresso#1]
y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.

[milk_latte#1]
espresso to five parts milk. A cappuccino uses the same espresso base but has equal parts milk and foam, giving it a lighter, airier texture.

---

Question: how is espresso made?


**Look at that carefully.** This is literally everything the LLM sees.

- Retrieval quality matters — wrong chunks → wrong answer
- Prompt structure matters — if you don't tell it to cite, it won't cite
- Chunk quality matters — fragmented chunks → fragmented context

---

## Cell 8 — Generate

Finally, ask the LLM. Plain chat completion. No framework.

### Why are we doing this?

In the previous cells, we built the individual parts of the RAG pipeline:

```text
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Similarity
    ↓
Top-K Retrieval
    ↓
Prompt Construction
Cell 8 connects these pieces and sends the retrieved context to the LLM so that it can generate the final answer.
What does ask_rag() do?
The ask_rag() function performs the complete RAG process for one question.
Question
    ↓
Retrieve top-K chunks
    ↓
Build prompt with retrieved context
    ↓
Send prompt to LLM
    ↓
Generate answer
    ↓
Return answer + sources + token usage
Step 1 — Retrieve
retrieved = retrieve(question, chunks, k=k)
The function retrieves the top-K chunks that are most similar to the question.
For our question:
how is espresso made?
the three retrieved chunks were:
coffee_espresso#0
coffee_espresso#1
milk_latte#1
Step 2 — Build the prompt
sys, user = build_prompt(question, retrieved, system=system)
The retrieved chunks are converted into the text prompt that will be sent to the LLM.
The prompt contains:
the system instruction
the retrieved context
the user's question
Step 3 — Generate the answer
resp = client.chat.completions.create(...)
The prompt is sent to gpt-4o-mini.
This is the generation part of RAG.
The LLM uses the retrieved context to formulate the answer.
Step 4 — Return useful information
The function returns:
question — the original question
answer — the generated answer
sources — the retrieved chunk IDs
tokens_in — number of prompt tokens
tokens_out — number of completion tokens
Result from this experiment
The generated answer was:
Espresso is made by forcing hot water under about 9 bars of pressure
through finely ground coffee beans. A single shot is typically 25 to
30 millilitres and takes 25 to 30 seconds to extract.
The answer cited:
[coffee_espresso#0]
which is the chunk containing the information used for the answer.
Token usage
The API call used:
Prompt tokens: 198
Completion tokens: 51
The prompt tokens represent information sent to the model, while completion tokens represent the generated answer.
Important observation
The retriever returned three chunks, but the final answer only needed information from one of them.
This shows that retrieval provides candidate context. The LLM decides which information from that context is useful for constructing the answer.
Position in the RAG pipeline
Documents
    ↓
Chunks
    ↓
Embeddings
    ↓
Question embedding
    ↓
Cosine similarity
    ↓
Top-K retrieval
    ↓
Prompt construction
    ↓
LLM generation  ← CURRENT STEP
    ↓
Answer + Sources
Key takeaway
RAG does not replace the LLM. Retrieval supplies relevant external context to the LLM, and the LLM uses that context to generate the answer.


In [15]:
def ask_rag(question, chunks, k=3, system=SYSTEM):
    retrieved = retrieve(question, chunks, k=k)
    sys, user = build_prompt(question, retrieved, system=system)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": sys},
            {"role": "user",   "content": user},
        ],
    )
    return {
        "question":   question,
        "answer":     resp.choices[0].message.content,
        "sources":    [hit["chunk_id"] for hit in retrieved],
        "tokens_in":  resp.usage.prompt_tokens,
        "tokens_out": resp.usage.completion_tokens,
    }

result = ask_rag(QUERY, all_chunks, k=3)

print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources retrieved: {result['sources']}")
print(f"Prompt tokens: {result['tokens_in']}  ·  Completion tokens: {result['tokens_out']}")

Q: how is espresso made?

A: Espresso is made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract [coffee_espresso#0].

Sources retrieved: ['coffee_espresso#0', 'coffee_espresso#1', 'milk_latte#1']
Prompt tokens: 198  ·  Completion tokens: 51


**Verify grounding.** Does the answer trace back to the retrieved chunks? Are
the sources cited in [brackets]?

If the answer says something NOT in the chunks, that's a **hallucination**.
It happens even to good systems — we'll see systematic examples in Day 2.

---

## Cell 8.5 — Change the system prompt

The system prompt is a knob. Let's see what changes when we use three
different prompts on the SAME retrieved chunks. Same question, same context,
different behaviour from the LLM.

This directly reinforces Topic 2 from the deck: the prompt is a stage in the
pipeline. It has failure modes just like chunking or retrieval.

### Why are we doing this?

In Cell 8, we used a system instruction that asked the LLM to answer only from the retrieved context.

Here, we run the same RAG question three times with three different system prompts:

1. STRICT
2. PERMISSIVE
3. NO INSTRUCTION

The purpose is to observe how system instructions affect the generated answer.

### Test question

```text
What is the ideal water temperature for oolong tea, and why?
Important observation
All three runs retrieved the same chunks:
tea_oolong#0
tea_black#0
tea_green#0
Therefore, the retrieval result did not change.
The main thing that changed was the instruction given to the LLM.
Same question
      ↓
Same retrieved context
      ↓
Different system instruction
      ↓
Different generated answer
STRICT
The strict system prompt instructed the model to:
answer only from the provided context
avoid using outside knowledge
say that the context does not contain the information if it is not explicitly stated
cite source IDs
The answer correctly reported the oolong brewing temperature as:
85 to 95 degrees Celsius
and cited:
[tea_oolong#0]
However, the explanation of why this temperature is suitable was not explicitly stated in the retrieved context.
This demonstrates an important limitation:
A strict system prompt does not guarantee perfect grounding. The LLM can still make inferences.
PERMISSIVE
The permissive prompt allowed the model to add helpful background knowledge.
The resulting answer provided a more detailed explanation involving the relationship between oolong, green, and black tea and how temperature affects flavour extraction.
This made the answer more informative, but some of the explanation was not explicitly present in the retrieved context.
Therefore:
More helpful background
        ↓
More informative answer
        ↓
Potentially less strictly grounded
NO INSTRUCTION
The third prompt only described the model as a helpful assistant.
The answer again supplied additional explanations that were not explicitly contained in the retrieved context.
This gives the model the greatest freedom to use its existing knowledge.
Comparison
Mode
Context restriction
Outside knowledge
Observation
STRICT
Strong
Discouraged
More grounded, but still capable of inference
PERMISSIVE
Moderate
Allowed
More detailed answer
NO INSTRUCTION
Very weak
Allowed
Most freedom for the LLM

Important RAG lesson
RAG has two important parts:
Retrieval
    ↓
What information do we give the LLM?

Generation
    ↓
How does the LLM use that information?
Retrieval and generation are therefore separate concerns.
In this experiment, retrieval remained the same while the system instruction changed. The generated answers changed accordingly.
Key takeaway
The retrieved context provides the evidence, while the system prompt influences how the LLM uses that evidence.
However, prompt instructions alone cannot guarantee that every generated statement is explicitly supported by the retrieved context.
This is one of the limitations we need to keep in mind when evaluating a naive RAG system.


In [16]:
SYSTEM_STRICT = (
    "You are a strict assistant. Answer ONLY from the context provided. "
    "If the answer is not explicitly stated in the context, respond with "
    "'The context does not contain this information.' Do not use any "
    "knowledge from your training. Cite the source id after each fact."
)

SYSTEM_PERMISSIVE = (
    "You are a helpful assistant. Use the context to answer, but you may "
    "add helpful background from what you know if it clarifies the answer."
)

SYSTEM_NO_INSTRUCTION = (
    "You are a helpful assistant."
)

test_query = "What is the ideal water temperature for oolong tea, and why?"

for label, sys_prompt in [
    ("STRICT     ", SYSTEM_STRICT),
    ("PERMISSIVE ", SYSTEM_PERMISSIVE),
    ("NO INSTRUCT", SYSTEM_NO_INSTRUCTION),
]:
    r = ask_rag(test_query, all_chunks, k=3, system=sys_prompt)
    print(f"── {label} ──")
    print(f"   {r['answer']}")
    print(f"   sources: {r['sources']}\n")

── STRICT      ──
   The ideal water temperature for oolong tea is 85 to 95 degrees Celsius. This temperature range is suitable for brewing oolong tea to achieve the desired strength and flavor, as it is partially oxidised, sitting between green and black tea in strength and colour. [tea_oolong#0]
   sources: ['tea_oolong#0', 'tea_black#0', 'tea_green#0']

── PERMISSIVE  ──
   The ideal water temperature for brewing oolong tea is between 85 to 95 degrees Celsius. This temperature range is suitable because oolong tea is partially oxidized, which means it requires a higher temperature than green tea (which is unoxidized) but lower than black tea (which is fully oxidized). Brewing oolong tea at this temperature helps to extract its complex flavors and aromas without causing bitterness, which can occur if the water is too hot or if the tea is steeped for too long.
   sources: ['tea_oolong#0', 'tea_black#0', 'tea_green#0']

── NO INSTRUCT ──
   The ideal water temperature for brewing oolong

**Discussion:**
- Did STRICT refuse to add anything beyond the context?
- Did PERMISSIVE mix context + training knowledge (potentially adding
  correct but ungrounded facts)?
- Did NO-INSTRUCTION follow the context at all, or start answering freely?

In production RAG systems, the SYSTEM prompt is versioned and tested. It's a
small piece of text with an outsized effect on system behaviour.

---

## Cell 9 — Break it (4 failure modes)

The best way to understand a system is to break it. Four deliberate failure
modes. Run each, look at the output, discuss what went wrong.

### 9a — Out-of-scope question

### What are we testing?

We deliberately ask a question about something that does not exist in our corpus:

```text
How do you brew beer?
Our corpus contains information about coffee, tea, chocolate, and latte, but it does not contain information about beer.
What happened during retrieval?
The cosine-similarity retriever still returned three chunks:
coffee_brewing#0
tea_black#0
coffee_brewing#1
This is a limitation of naive retrieval.
The retriever compares the question with the chunks that exist and returns the closest matches. It does not automatically know that none of the chunks actually answer the question.
Conceptually:
Question about beer
       ↓
No beer information exists
       ↓
Cosine similarity
       ↓
Closest available chunks
       ↓
Coffee / tea chunks
Therefore, retrieval was not ideal.
What happened during generation?
The LLM answered:
The provided context does not contain any information about brewing beer.
This was the correct grounded behavior.
The model did not use its general knowledge to provide instructions for brewing beer.
What does this demonstrate?
This example shows that retrieval and generation can fail independently.
Retrieval
    ↓
Returned irrelevant chunks
    ❌

Generation
    ↓
Recognised that the context did not contain the answer
    ✅
The system prompt from Cell 7 helped the LLM avoid answering from its general knowledge.
Key takeaway
Naive cosine retrieval can return the closest available chunks even when the corpus does not contain the answer. A grounding instruction can still prevent the LLM from hallucinating an answer.
However, the retrieval stage itself needs a way to recognize when there is no sufficiently relevant result. This is one of the limitations of naive RAG.


In [17]:
r = ask_rag("how do you brew beer?", all_chunks, k=3)
print(f"Q: {r['question']}\n")
print(f"A: {r['answer']}\n")
print(f"Sources retrieved: {r['sources']}")
# Discussion:
#   - Cosine still returned SOMETHING. Look at the sources.
#   - Did the LLM refuse (correct)?
#   - Or did it try to answer from training data (violates our grounding rule)?

Q: how do you brew beer?

A: The provided context does not contain any information about brewing beer.

Sources retrieved: ['coffee_brewing#0', 'tea_black#0', 'coffee_brewing#1']


### 9b — No context at all (k=0 equivalent)

What if we retrieve nothing? Test whether the system prompt actually holds.

### What are we testing?

In this experiment, we deliberately provide the LLM with **no retrieved context**.

The question is:

```text
How is espresso made?
Normally, our RAG pipeline would retrieve relevant espresso chunks before calling the LLM.
Here, we pass an empty list:
[]
Therefore, the generated prompt contains the question but no supporting information.
What did the user message contain?
Context:


---

Question: how is espresso made?
The context section is completely empty.
What happened?
The LLM responded:
The provided context does not contain information on how espresso is made.
This is the expected grounded behavior.
Although the language model may have general knowledge about espresso, it followed the system instruction to answer only from the supplied context.
Pipeline
Question
    ↓
No retrieved chunks
    ↓
Empty context
    ↓
Grounding instruction
    ↓
LLM
    ↓
Refuses to answer
Comparison with Cell 9a
In Cell 9a, the retriever returned irrelevant chunks because the corpus did not contain information about beer.
In this cell, there are no chunks at all.
However, in both cases the LLM correctly refused to use its outside knowledge.
Key takeaway
Grounding instructions can prevent the LLM from using its own knowledge when the required information is not present in the supplied context.
However, this does not solve the retrieval problem. A naive retriever can still return irrelevant chunks, as demonstrated in Cell 9a.


In [18]:
sys_msg, user_msg = build_prompt("how is espresso made?", [])
print("═══ USER MESSAGE WITH ZERO CONTEXT ═══")
print(user_msg)
print()

resp = client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0.0,
    messages=[
        {"role": "system", "content": sys_msg},
        {"role": "user",   "content": user_msg},
    ],
)
print("═══ ANSWER ═══")
print(resp.choices[0].message.content)
# Discussion: did the LLM follow the rule (refuse) or leak training knowledge?

═══ USER MESSAGE WITH ZERO CONTEXT ═══
Context:


---

Question: how is espresso made?



═══ ANSWER ═══
The provided context does not contain information on how espresso is made.


### 9c — Garbled query

### What are we testing?

The question is deliberately written incorrectly:

```text
expresso how mak??

It contains a spelling mistake and incomplete wording.
The purpose is to see whether the embedding-based retrieval system can still identify the intended topic.
What happened during retrieval?
The retriever returned:
coffee_espresso#0
coffee_espresso#1
milk_latte#1

The first two results are the espresso chunks.
This shows that semantic embeddings can tolerate imperfect wording. The query did not need to exactly match the word espresso for the relevant chunks to be retrieved.
Conceptually:
Poorly written query
        ↓
Question embedding
        ↓
Semantic similarity
        ↓
Espresso-related chunks

What happened during generation?
The LLM responded:
The context does not provide specific instructions on how to make espresso.
It only describes what espresso is and its characteristics.

The answer stayed within the supplied context and did not invent additional instructions from the model's general knowledge.
However, the retrieved context did contain some information about how espresso is made, including the pressure, finely ground beans, extraction time, and shot volume.
Therefore, the model was somewhat conservative in interpreting the incomplete question.
What does this demonstrate?
This test shows two different properties:
Embedding strength:
Typos and poor wording
        ↓
Relevant semantic representation
        ↓
Relevant chunks retrieved

Generation limitation:
Relevant context available
        ↓
LLM interprets the incomplete question cautiously
        ↓
May refuse even when partial information is available

Key takeaway
Embeddings can be robust to spelling mistakes and imperfect queries, which is a major advantage of semantic retrieval. However, good retrieval does not automatically guarantee that the LLM will make the best use of the retrieved context.

In [19]:
r = ask_rag("expresso how mak??", all_chunks, k=3)
print(f"Q: {r['question']}\n")
print(f"A: {r['answer']}\n")
print(f"Sources retrieved: {r['sources']}")
# Discussion: embeddings tolerate typos surprisingly well. Retrieval likely
# still found the coffee chunks. This is embeddings' strength.

Q: expresso how mak??

A: The context does not provide specific instructions on how to make espresso. It only describes what espresso is and its characteristics.

Sources retrieved: ['coffee_espresso#0', 'coffee_espresso#1', 'milk_latte#1']


### 9d — Partial-match query (mixed relevance)

This one's more subtle. Ask a question where SOME retrieved chunks are
relevant and some aren't. What does the LLM do with mixed context?

### What are we testing?

The question asks for multiple related facts:

```text
Compare the caffeine content of black tea and green tea,
and the water temperatures used to brew them.
To answer completely, the system needs evidence covering both tea types and both requested aspects.
What did Top-K retrieval return?
With:
k = 3
the retriever returned:
1. tea_black#1   cosine = 0.618
2. tea_green#0   cosine = 0.534
3. tea_green#1   cosine = 0.525
The important missing chunk was:
tea_black#0
This chunk contains the brewing temperature for black tea.
What information was available?
The retrieved context contained:
Black tea caffeine:
Black tea typically contains more caffeine than green tea.
from:
tea_black#1
It also contained:
Green tea temperature:
70 to 80 degrees Celsius
from:
tea_green#0
However, the retrieved context did not contain the black tea brewing temperature.
What did the LLM do?
The LLM answered the supported parts:
Black tea typically contains more caffeine than green tea.
and:
Green tea is steeped at around 70 to 80 degrees Celsius.
For the missing information, it correctly stated:
The context does not provide the specific water temperature
for brewing black tea.
Therefore, the LLM did not invent the missing value.
What failed?
The retrieval stage was incomplete.
id="rjv0p4"
Question
   ↓
Needs several facts
   ↓
Top-K = 3
   ↓
Only some required chunks retrieved
   ↓
Incomplete context
   ↓
LLM gives incomplete answer
Important distinction
This is not primarily a generation failure.
The LLM could not provide the missing black-tea temperature because that information was not present in the retrieved context.
Therefore:
Retrieval       ❌ Incomplete
Grounding       ✅ Good
Hallucination   ❌ No
Key takeaway
A multi-fact question can fail when Top-K retrieval does not retrieve all the chunks needed to answer every part of the question.
The LLM can only work with the evidence that reaches its prompt.
Simply having the information somewhere in the corpus is not enough — the retrieval stage must actually retrieve it.


In [20]:
r = ask_rag(
    "Compare the caffeine content of black tea and green tea, and the water "
    "temperatures used to brew them.",
    all_chunks, k=3,
)
print(f"Q: {r['question']}\n")
print(f"A: {r['answer']}\n")
print(f"Sources retrieved: {r['sources']}\n")
print("Retrieved chunks (with content):\n")
for hit in retrieve(r['question'], all_chunks, k=3):
    print(f"  [{hit['chunk_id']}] cosine={hit['score']:.3f}")
    print(f"    {hit['text'][:120]}...\n")
# Discussion:
#   - This is a multi-fact question. Does k=3 give us BOTH tea chunks?
#   - If not, the LLM has half the answer. Did it fabricate the other half?
#   - This is a common naive-RAG failure — top-3 doesn't cover both facts.

Q: Compare the caffeine content of black tea and green tea, and the water temperatures used to brew them.

A: Black tea typically contains more caffeine than green tea [tea_black#1]. Green tea is steeped in water at around 70 to 80 degrees Celsius [tea_green#0]. The context does not provide the specific water temperature for brewing black tea.

Sources retrieved: ['tea_black#1', 'tea_green#0', 'tea_green#1']

Retrieved chunks (with content):

  [tea_black#1] cosine=0.618
    . Popular varieties include Assam, Darjeeling, and Ceylon. Black tea typically contains more caffeine than green tea....

  [tea_green#0] cosine=0.534
    Green tea is made from unoxidised leaves of Camellia sinensis. It is steeped in water at around 70 to 80 degrees Celsius...

  [tea_green#1] cosine=0.525
    or longer steeping produces a bitter, astringent cup. Green tea is high in an antioxidant called EGCG....



**These four failure modes are the argument for measurement.** You cannot
eyeball a RAG system's reliability. You need to run enough queries, grade
them, and track the numbers over time.

**Day 2 is where we do that** — take this pipeline and stress-test it
systematically.

---

## Recap — what you built (Day 1)

In ~100 lines of Python you have a complete naive RAG pipeline:

| Stage | Function | What it does |
|---|---|---|
| Chunk | `chunk_text` | Split docs into fixed-size windows |
| Embed | `embed_batch` | Text → 1536-dim vectors via OpenAI |
| Similarity | `cosine` | Vector distance metric |
| Retrieve | `retrieve` | Rank chunks, pick top-K |
| Prompt | `build_prompt` | System + context + question |
| Generate | `ask_rag` | Chat completion → answer + sources |

**Next up (Day 2):** we take this same pipeline and turn every knob to see
how each one affects results. Chunk size, K, embedding model, system prompt.

**After that (Track B, take-home):** we apply this pattern to your capstone.
See `AI-RAG_W6_Application_Growth_Guide.md`.
